# 🏆 KIW 모의투자대회 - 종목 스크리닝 (정량 필터)

**목표 수익률**: 30% (2026.04.01 ~ 05.31, 2개월)

**대회 규칙 반영**:
- 종목별 15% 한도 (삼성전자 40% 예외)
- 시총 1조 미만 합산 30% 이하
- 5일 평균 거래대금 30억 이상만 매수 가능
- 주간 회전율 5% 이상 유지

**스크리닝 기준**:
| 지표 | 조건 | 목적 |
|------|------|------|
| EPS | > 0 | 흑자 기업 필터 |
| 부채비율 | < 200% | 재무 안정성 |
| ROE | > 10% | 자본 효율성 |

## 0. 라이브러리 설치 & 임포트

In [ ]:
# 최초 1회 설치
# !pip install finance-datareader pandas numpy openpyxl

import FinanceDataReader as fdr
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f"실행일: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"대회 기간: 2026.04.01 ~ 2026.05.31")

## 1. KOSPI + KOSDAQ 전종목 로드

In [ ]:
# KRX 전종목 리스트 (KOSPI + KOSDAQ)
kospi = fdr.StockListing('KOSPI')
kosdaq = fdr.StockListing('KOSDAQ')

# 마켓 태그 추가
kospi['Market'] = 'KOSPI'
kosdaq['Market'] = 'KOSDAQ'

stocks = pd.concat([kospi, kosdaq], ignore_index=True)

print(f"KOSPI: {len(kospi)}종목 | KOSDAQ: {len(kosdaq)}종목 | 합계: {len(stocks)}종목")
stocks.head(3)

In [ ]:
# 컬럼 확인 (FDR 버전에 따라 컬럼명이 다를 수 있음)
print("사용 가능한 컬럼:")
print(stocks.columns.tolist())

## 2. 기본 전처리 & 대회 제약 필터

In [ ]:
# ============================================================
# 컬럼명 매핑 (FDR 버전별 대응)
# FDR이 반환하는 컬럼명을 확인 후 아래 매핑을 조정하세요
# ============================================================

# 예상 컬럼명 후보 (FDR 버전에 따라 다름)
col_candidates = {
    'code':   ['Code', 'Symbol', 'ISU_SRT_CD', 'code', 'symbol'],
    'name':   ['Name', 'ISU_ABBRV', 'name', '종목명'],
    'marcap': ['Marcap', 'MKTCAP', 'marcap', '시가총액'],
    'close':  ['Close', 'close', 'TDD_CLSPRC', '종가'],
    'volume': ['Volume', 'volume', 'ACC_TRDVOL', '거래량'],
}

def find_col(df, candidates):
    """후보 컬럼명 중 실제 존재하는 컬럼 반환"""
    for c in candidates:
        if c in df.columns:
            return c
    return None

col_code   = find_col(stocks, col_candidates['code'])
col_name   = find_col(stocks, col_candidates['name'])
col_marcap = find_col(stocks, col_candidates['marcap'])
col_close  = find_col(stocks, col_candidates['close'])

print(f"매핑된 컬럼 → 코드: {col_code}, 이름: {col_name}, 시총: {col_marcap}, 종가: {col_close}")

# 통일된 컬럼명으로 리네임
rename_map = {}
if col_code:   rename_map[col_code] = 'Code'
if col_name:   rename_map[col_name] = 'Name'
if col_marcap: rename_map[col_marcap] = 'Marcap'
if col_close:  rename_map[col_close] = 'Close'

stocks = stocks.rename(columns=rename_map)
print(f"\n리네임 후 컬럼: {stocks.columns.tolist()[:10]}...")

In [ ]:
# ============================================================
# 대회 필터 1: 시총 구간 태깅
# ============================================================

if 'Marcap' in stocks.columns:
    stocks['Marcap_억'] = stocks['Marcap'] / 1e8
    stocks['시총구간'] = pd.cut(
        stocks['Marcap_억'],
        bins=[0, 1000, 5000, 10000, 50000, float('inf')],
        labels=['~1천억(소형)', '1천~5천억(중소형)', '5천~1조(중형)', '1조~5조(중대형)', '5조+(대형)']
    )
    stocks['소형주_여부'] = stocks['Marcap'] < 1e12  # 시총 1조 미만 → 대회 합산 30% 제한
    
    print("시총 구간별 종목 수:")
    print(stocks['시총구간'].value_counts().sort_index())
else:
    print("⚠️ 시가총액 컬럼을 찾을 수 없습니다. stocks.columns를 확인하세요.")

## 3. 거래대금 필터 (5일 평균 30억 이상)

In [ ]:
# ============================================================
# 5일 평균 거래대금 계산
# FDR의 StockListing에 거래대금이 없으면 개별 조회 필요
# → 전종목 개별 조회는 시간이 오래 걸리므로, 
#   StockListing 데이터에 거래대금이 있으면 바로 사용
# ============================================================

MIN_TRADING_VALUE = 30e8  # 30억원

# FDR StockListing에 'Amount' 또는 'Turnover' 등 거래대금 컬럼이 있는지 확인
trading_col_candidates = ['Amount', 'amount', 'ACC_TRDVAL', '거래대금', 'Turnover']
col_amount = find_col(stocks, trading_col_candidates)

if col_amount:
    stocks = stocks.rename(columns={col_amount: 'TradingValue'})
    stocks['거래대금_억'] = stocks['TradingValue'] / 1e8
    eligible = stocks[stocks['TradingValue'] >= MIN_TRADING_VALUE].copy()
    print(f"거래대금 30억+ 종목: {len(eligible)}개 (전체 {len(stocks)}개 중)")
else:
    print("⚠️ 거래대금 컬럼 없음 → 개별 종목 최근 5일 데이터 조회 필요")
    print("아래 셀에서 샘플링 방식으로 거래대금을 계산합니다.")
    eligible = stocks.copy()  # 일단 전체 유지

In [ ]:
# ============================================================
# [선택] 거래대금 컬럼이 없을 경우: 개별 종목 5일 조회
# 전종목 조회 시 ~30분 소요 → 시총 상위 500개만 먼저 필터링
# ============================================================

FETCH_INDIVIDUAL = False  # True로 변경하면 실행

if FETCH_INDIVIDUAL and col_amount is None:
    from tqdm.notebook import tqdm
    
    # 시총 상위 500개만 대상
    if 'Marcap' in stocks.columns:
        target = stocks.nlargest(500, 'Marcap')
    else:
        target = stocks.head(500)
    
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=10)).strftime('%Y-%m-%d')
    
    trading_values = {}
    errors = []
    
    for _, row in tqdm(target.iterrows(), total=len(target), desc="거래대금 조회"):
        code = row['Code']
        try:
            df = fdr.DataReader(code, start_date, end_date)
            if len(df) >= 5:
                # 거래대금 = 종가 × 거래량 (근사치)
                df['TradingValue'] = df['Close'] * df['Volume']
                avg_5d = df['TradingValue'].tail(5).mean()
                trading_values[code] = avg_5d
        except:
            errors.append(code)
    
    tv_series = pd.Series(trading_values, name='AvgTradingValue5D')
    target = target.set_index('Code')
    target['AvgTradingValue5D'] = tv_series
    target = target.reset_index()
    
    eligible = target[target['AvgTradingValue5D'] >= MIN_TRADING_VALUE].copy()
    eligible['거래대금_억'] = eligible['AvgTradingValue5D'] / 1e8
    
    print(f"조회 완료: {len(trading_values)}개 성공, {len(errors)}개 실패")
    print(f"거래대금 30억+ 종목: {len(eligible)}개")

## 4. 재무 지표 스크리닝 (EPS, 부채비율, ROE)

In [ ]:
# ============================================================
# FDR에서 재무 데이터 가져오기
# fdr.StockListing에 이미 포함된 경우도 있고,
# 별도 조회가 필요할 수도 있음
# ============================================================

# 재무지표 컬럼 존재 여부 확인
fin_cols = {
    'eps': ['EPS', 'eps'],
    'roe': ['ROE', 'roe'],
    'debt_ratio': ['DebtRatio', 'debt_ratio', '부채비율', 'DPS'],
    'per': ['PER', 'per'],
    'pbr': ['PBR', 'pbr'],
}

found_fins = {}
for key, candidates in fin_cols.items():
    col = find_col(eligible, candidates)
    if col:
        found_fins[key] = col

print(f"StockListing에서 발견된 재무 컬럼: {found_fins}")

if not found_fins:
    print("\n⚠️ 재무지표가 StockListing에 없습니다.")
    print("→ 아래 셀에서 수동 계산 또는 외부 데이터 병합이 필요합니다.")

In [ ]:
# ============================================================
# 스크리닝 파라미터 (여기서 조정 가능)
# ============================================================

SCREEN_PARAMS = {
    'eps_min':        0,      # EPS > 0 (흑자)
    'debt_ratio_max': 200,    # 부채비율 < 200%
    'roe_min':        10,     # ROE > 10%
}

print("=" * 50)
print("📋 스크리닝 기준")
print("=" * 50)
for k, v in SCREEN_PARAMS.items():
    print(f"  {k}: {v}")

In [ ]:
# ============================================================
# 스크리닝 적용
# ============================================================

screened = eligible.copy()
filters_applied = []

# EPS 필터
if 'eps' in found_fins:
    col = found_fins['eps']
    screened[col] = pd.to_numeric(screened[col], errors='coerce')
    before = len(screened)
    screened = screened[screened[col] > SCREEN_PARAMS['eps_min']]
    filters_applied.append(f"EPS > {SCREEN_PARAMS['eps_min']}: {before} → {len(screened)}")

# 부채비율 필터
if 'debt_ratio' in found_fins:
    col = found_fins['debt_ratio']
    screened[col] = pd.to_numeric(screened[col], errors='coerce')
    before = len(screened)
    screened = screened[screened[col] < SCREEN_PARAMS['debt_ratio_max']]
    filters_applied.append(f"부채비율 < {SCREEN_PARAMS['debt_ratio_max']}%: {before} → {len(screened)}")

# ROE 필터
if 'roe' in found_fins:
    col = found_fins['roe']
    screened[col] = pd.to_numeric(screened[col], errors='coerce')
    before = len(screened)
    screened = screened[screened[col] > SCREEN_PARAMS['roe_min']]
    filters_applied.append(f"ROE > {SCREEN_PARAMS['roe_min']}%: {before} → {len(screened)}")

print("=" * 50)
print("🔍 필터링 결과")
print("=" * 50)
for f in filters_applied:
    print(f"  ✅ {f}")

if not filters_applied:
    print("  ⚠️ 적용된 필터 없음 - 재무 컬럼이 부족합니다.")
    print("  → 아래 '수동 재무 데이터 병합' 셀을 활용하세요.")

print(f"\n최종 스크리닝 통과: {len(screened)}종목")

## 5. [대안] FDR에 재무데이터 없을 때 - 수동 병합

In [ ]:
# ============================================================
# FDR StockListing에 EPS/ROE/부채비율이 없는 경우
# → KRX에서 CSV 다운로드 후 병합하는 방식
# 
# 1) data.krx.co.kr → 기본통계 → 주식 → 세부안내
#    → PER/PBR/배당수익률 데이터 다운로드
# 2) DART OpenAPI로 재무제표 조회 (API키 필요)
# 3) 증권사 HTS에서 전종목 재무지표 CSV 추출
# ============================================================

# 예시: CSV 파일 병합
# fin_data = pd.read_csv('krx_financial_data.csv', encoding='cp949')
# screened = eligible.merge(fin_data, on='Code', how='left')

# 예시: 수동 입력 (관심 종목이 정해진 경우)
manual_financials = pd.DataFrame({
    'Code':       ['005930', '000660', '012330', '009540', '042700'],
    'Name_check': ['삼성전자', 'SK하이닉스', '현대모비스', '한국조선해양', '한미반도체'],
    'EPS':        [4000,     30000,      25000,     15000,     5000],
    'ROE':        [8.5,      25.0,       12.0,      18.0,      22.0],
    'DebtRatio':  [30,       50,         80,        120,       40],
})

print("수동 입력 예시 (실제 데이터로 교체 필요):")
manual_financials

## 6. 대회 규칙 적합성 태깅 & 최종 결과

In [ ]:
# ============================================================
# 대회 규칙 기반 포트폴리오 적합성 태깅
# ============================================================

def tag_kiw_constraints(df):
    """KIW 대회 규칙에 따른 종목별 제약 태깅"""
    result = df.copy()
    
    # 종목별 최대 비중
    result['최대비중'] = result['Code'].apply(
        lambda x: '40%' if x == '005930' else '15%'
    )
    
    # 소형주 태깅 (시총 1조 미만 → 합산 30% 제한)
    if 'Marcap' in result.columns:
        result['소형주제한'] = result['Marcap'].apply(
            lambda x: '⚠️ 소형주(합산30%제한)' if x < 1e12 else '✅ 제한없음'
        )
    
    return result

final = tag_kiw_constraints(screened)

# 디스플레이 컬럼 선택
display_cols = ['Code', 'Name', 'Market']
if 'Close' in final.columns: display_cols.append('Close')
if 'Marcap_억' in final.columns: display_cols.append('Marcap_억')
if '시총구간' in final.columns: display_cols.append('시총구간')
if '거래대금_억' in final.columns: display_cols.append('거래대금_억')
for key in ['eps', 'roe', 'debt_ratio', 'per', 'pbr']:
    if key in found_fins:
        display_cols.append(found_fins[key])
display_cols.extend(['최대비중', '소형주제한'])

# 존재하는 컬럼만 필터
display_cols = [c for c in display_cols if c in final.columns]

print(f"\n{'='*60}")
print(f"📊 최종 스크리닝 결과: {len(final)}종목")
print(f"{'='*60}")

# 시총 기준 상위 30개 표시
if 'Marcap' in final.columns:
    top30 = final.nlargest(30, 'Marcap')[display_cols]
else:
    top30 = final.head(30)[display_cols]

top30

## 7. 스크리닝 결과 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'  # 한글 폰트
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('KIW 스크리닝 결과 분포', fontsize=14, fontweight='bold')

# 1) 시총 구간별 분포
if '시총구간' in final.columns:
    final['시총구간'].value_counts().sort_index().plot(
        kind='barh', ax=axes[0], color='#2196F3'
    )
    axes[0].set_title('시총 구간별 종목 수')
    axes[0].set_xlabel('종목 수')

# 2) 마켓별 분포
if 'Market' in final.columns:
    final['Market'].value_counts().plot(
        kind='pie', ax=axes[1], autopct='%1.0f%%', colors=['#4CAF50', '#FF9800']
    )
    axes[1].set_title('KOSPI vs KOSDAQ')
    axes[1].set_ylabel('')

# 3) ROE 분포 (있을 경우)
if 'roe' in found_fins and found_fins['roe'] in final.columns:
    roe_col = found_fins['roe']
    final[roe_col].dropna().plot(
        kind='hist', bins=20, ax=axes[2], color='#9C27B0', edgecolor='white'
    )
    axes[2].axvline(x=SCREEN_PARAMS['roe_min'], color='red', linestyle='--', label=f"기준선 {SCREEN_PARAMS['roe_min']}%")
    axes[2].set_title('ROE 분포')
    axes[2].set_xlabel('ROE (%)')
    axes[2].legend()

plt.tight_layout()
plt.show()

## 8. 결과 저장 (CSV / Excel)

In [ ]:
# ============================================================
# 스크리닝 결과 파일 저장
# ============================================================

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
filename = f'kiw_screening_{timestamp}'

# CSV 저장
final[display_cols].to_csv(f'{filename}.csv', index=False, encoding='utf-8-sig')
print(f"✅ CSV 저장: {filename}.csv")

# Excel 저장 (시트 분리: 전체 + 소형주 + 대형주)
with pd.ExcelWriter(f'{filename}.xlsx', engine='openpyxl') as writer:
    final[display_cols].to_excel(writer, sheet_name='전체', index=False)
    
    if 'Marcap' in final.columns:
        large = final[final['Marcap'] >= 1e12][display_cols]
        small = final[final['Marcap'] < 1e12][display_cols]
        large.to_excel(writer, sheet_name='대형주(1조+)', index=False)
        small.to_excel(writer, sheet_name='소형주(1조미만)', index=False)

print(f"✅ Excel 저장: {filename}.xlsx")
print(f"\n총 {len(final)}종목 저장 완료")

---
## 📝 Next Steps

**정량 분석 확장:**
- PER/PBR 밸류에이션 필터 추가
- 모멘텀 지표 (3M/6M 수익률) 오버레이
- 거래대금 추세 (증가/감소) 분석

**정성 분석 연계:**
- 스크리닝 통과 종목 → 테마/섹터 매핑
- 증권사 리서치 컨센서스 매칭
- 이벤트 캘린더 (실적발표, 정책 등) 오버레이

**포트폴리오 구성:**
- 대회 규칙 내 최적 비중 배분
- 회전율 5% 충족 시뮬레이션
- 리스크 시나리오 (최대 낙폭) 백테스트